# 9 错误、调试与测试

## 9.1 错误处理

程序运行过程中可能发生错误，Python 使用异常表示这些错误，可以用 try...except 捕获并处理

try 中放可能出错的代码；except 处理指定类型的异常；else 在 try 没有异常时执行；finally 无论是否发生异常都会执行

In [ ]:
try:
    print('try...')
    r = 10 / int('2')
    print('result:', r)
except ValueError as e:
    print('ValueError:', e)
except ZeroDivisionError as e:
    print('ZeroDivisionError:', e)
else:
    print('no error!')
finally:
    print('finally...')
print('END')

使用try...except捕获错误的好处是可以跨越多层调用

In [ ]:
def foo(s):
    return 10 / int(s)

def bar(s):
    return foo(s) * 2

def main():
    try:
        bar('0')
    except Exception as e:
        print('Error:', e)
    finally:
        print('finally...')

main()


调用栈：

如果错误没有被捕获，它就会一直往上抛，最后被Python解释器捕获，打印一个错误信息，然后程序退出

下面的示例会故意触发 ZeroDivisionError，用于观察调用栈。后面的抛出错误、调试示例也会故意报错，适合逐个运行单元格观察；异常发生后，可以继续手动运行后面的单元格

In [ ]:
# err.py:
def foo(s):
    return 10 / int(s)

def bar(s):
    return foo(s) * 2

def main():
    bar('0')

main()

记录错误：

In [ ]:
import logging

def foo(s):
    return 10 / int(s)

def bar(s):
    return foo(s) * 2

def main():
    try:
        bar('0')
    except Exception as e:
        logging.exception(e)

main()
print('END')

抛出错误：


In [ ]:
class FooError(ValueError):
    pass

def foo(s):
    n = int(s)
    if n==0:
        raise FooError('invalid value: %s' % s)
    return 10 / n

foo('0')

## 9.2 调试

第一种方法，用 print 打印可能有问题的变量

In [ ]:
def foo(s):
    n = int(s)
    print('>>> n = %d' % n)
    return 10 / n

def main():
    foo('0')

main()

第二种方法，使用 assert

断言用于检查调试时的假设，条件不成立时会抛出 AssertionError。使用 python -O 运行时，assert 会被忽略，因此不能代替必要的参数检查

In [ ]:
def foo(s):
    n = int(s)
    assert n != 0, 'n is zero!'
    return 10 / n

def main():
    foo('0')

main()

第三种方法，使用 logging

In [ ]:
import logging

s = '0'
n = int(s)
logging.info('n = %d' % n)
print(10 / n)


In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

s = '0'
n = int(s)
logging.info('n = %d' % n)
print(10 / n)


使用 pdb，在可能出错的地方放一个pdb.set_trace()

这个示例会进入交互式调试，适合单独运行。n 执行下一行，p 变量名 查看变量，c 继续运行，q 退出调试；批量运行笔记时可以跳过这个单元格

In [ ]:
# err.py
import pdb

s = '0'
n = int(s)
pdb.set_trace() # 运行到这里会自动暂停
print(10 / n)

## 9.3 单元测试

单元测试就是对一个函数、方法等小单元进行检查，确认实际结果符合预期

Python 标准库 unittest 提供了测试框架：测试类继承 unittest.TestCase，测试方法的名称以 test_ 开头

既要测试正常输入，也要测试边界值和错误输入

In [ ]:
def myabs(x):
    if not isinstance(x, (int, float)):
        raise TypeError('bad operand type')
    if x >= 0:
        return x
    return -x

In [ ]:
import unittest

class TestMyabs(unittest.TestCase):
    def test_positive(self):
        self.assertEqual(myabs(3), 3)

    def test_negative(self):
        self.assertEqual(myabs(-3), 3)

    def test_zero(self):
        self.assertEqual(myabs(0), 0)

    def test_type_error(self):
        with self.assertRaises(TypeError):
            myabs('3')

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestMyabs)
result = unittest.TextTestRunner(verbosity=2).run(suite)
print('测试通过:', result.wasSuccessful())

assertEqual(a, b) 检查两个值是否相等；assertTrue(x) 检查条件是否为真；assertRaises(异常类型) 检查是否抛出了预期的异常

上面显式加载测试类，适合在 Notebook 中重复运行。如果保存为 test_myabs.py，也可以在终端中运行 python3 -m unittest test_myabs.py

需要为每个测试准备数据时，可以定义 setUp()；测试后的清理工作可以放在 tearDown() 中，它们会围绕每个测试方法执行

## 9.4 文档测试

文档测试把调用示例和预期输出写在函数的文档字符串中，再使用 doctest 检查

>>> 后面是要执行的代码，下一行是预期输出。文档字符串需要放在函数体的开头

In [ ]:
def square(n):
    """返回一个数的平方。

    >>> square(3)
    9
    >>> square(0)
    0
    >>> square(-2)
    4
    """
    return n * n

In [ ]:
import doctest

# 只运行这个函数文档中的示例，适合在Notebook中使用
doctest.run_docstring_examples(square, globals(), name='square', verbose=True)

如果代码保存在 .py 文件中，可以使用 doctest.testmod() 检查整个模块，或者在终端中运行 python3 -m doctest -v 文件名.py

输出中的空格、换行等也会参与比较，示例的预期输出应与实际输出保持一致

Practice: 给 myabs() 增加一个小数输入的单元测试，再给 square() 增加一个新的文档测试示例

参考：[unittest](https://docs.python.org/3.11/library/unittest.html)、[doctest](https://docs.python.org/3.11/library/doctest.html)